# 05 — Mixed sources

**Scenario.** Real internal knowledge is not one wiki. It is policy markdown, PDF runbooks, CSV matrices, React screens, HTML FAQ, and JSON org charts. RAG has to ingest all of them into the same index.

1. List every file under `data/` with its type and classification (from `manifest.yaml`).
2. Ask a question that is only answered in a CSV, then one only answered in a PDF.

Context:

- Real enterprises store knowledge across many systems, not one clean wiki.
- Each loader normalizes content into the same `Document` shape.
- `manifest.yaml` centralizes metadata for formats that cannot carry frontmatter.

Edge cases:

- PDF extraction can lose reading order.
- CSV rows need column names preserved.
- Source files may contain examples, comments, or test fixtures that look like real facts.


Definitions and context:

| Source | RAG concern |
| --- | --- |
| Markdown | Headings help create clean sections |
| PDF | Extraction can reorder text or drop table structure |
| CSV | Rows need column names to remain meaningful |
| TSX | Comments and fixtures may contain policy-relevant text |
| HTML | Pages may include navigation, boilerplate, or hostile text |
| JSON/YAML | Structured data must be rendered into searchable text |

The loader's job is normalization: every format becomes a `Document`, then a
set of chunks. The original path and source type are still kept for citations,
debugging, and audit.


In [ ]:
import _course
from lib.ingest import load_documents
from lib.pipeline import build_index, retrieve

docs = load_documents()
print(f"{'type':6} {'class':14} {'status':11} path")
for d in docs:
    print(f"{d.source_type:6} {d.classification:14} {d.status:11} {d.source_path}")


In [ ]:
index = build_index()

def peek(question, role="employee"):
    print("Q:", question)
    for h in retrieve(index, question, role, k=4):
        print(f"  {h.chunk.source_type:5} {h.chunk.doc_id:22} {h.chunk.section}")
    print()

peek("How long do we keep retrieval audit logs?")
peek("What is SEV-1 for a full account number in an unsanctioned tool?")


`manifest.yaml` is the ACL source of truth so a PDF can be Restricted even though it has no YAML frontmatter. Next: `06-hybrid-search.ipynb`.
